# Week 7: Project Studio Exercises
## Ensembling and Intermittent Demand

**Objective.** Two tools for the last stretch of the final project: combine models so that you stop betting on one of them, and forecast series that are mostly zeros without being fooled by the scoring rule.

### How this notebook works

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | Average several models, and see what the average buys and what it does not. |
| 2 | Blanks we fill in together | Weighted ensembles, scored honestly on windows the weights never saw. |
| 3 | Walkthrough | Intermittent demand: ADI, CV², Croston's family, and why MAE rewards forecasting zero. |
| 4 | On your own, ~15 min | Ensemble the intermittent models, handle a discontinued item, and turn a forecast into a stocking level. |

Everything here runs on a built-in or simulated series, so nothing needs to be downloaded except AirPassengers.

In [ ]:
!pip install -q statsforecast pandas numpy plotly

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from statsforecast import StatsForecast
from statsforecast.models import (
    AutoETS, AutoARIMA, AutoTheta, SeasonalNaive, Naive,
    CrostonClassic, CrostonSBA, TSB, ADIDA, IMAPA, HistoricAverage,
)

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = (pd.read_csv(URL, parse_dates=["Month"])
         .rename(columns={"Month": "ds", "Passengers": "y"})
         .assign(unique_id="airline"))

m = 12
air.tail(2)

---
## Part 1. The plain average

Different model families have different blind spots, so their errors are only partly correlated, and averaging cancels some of the noise. The simplest recipe is the equal-weight mean, and it is hard to beat.

We score every model on the **same** rolling origins: 6 windows, 12 months ahead, stepping 6 months. `Naive` (no seasonality) is in the pool on purpose. It is a weak model, and we will see what it does to an average.

In [ ]:
models = [
    AutoETS(season_length=m),
    AutoARIMA(season_length=m),
    AutoTheta(season_length=m),
    Naive(),
]
names = ["AutoETS", "AutoARIMA", "AutoTheta", "Naive"]
good = names[:3]

sf = StatsForecast(models=models, freq="MS")
cv = sf.cross_validation(df=air[["unique_id", "ds", "y"]], h=12, n_windows=6, step_size=6)

cv["Avg3"] = cv[good].mean(axis=1)       # average of the three reasonable models
cv["Avg4"] = cv[names].mean(axis=1)      # ... plus the weak one

cols = names + ["Avg3", "Avg4"]
err = cv[cols].sub(cv["y"], axis=0)
score = pd.DataFrame({"MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean())}).round(1)
score

In [ ]:
# How differently are they wrong?  Correlation of errors, good models only.
err[good].corr().round(2)

**Reading the table.**

- The average usually will not beat the best single model on one series. What it buys is **insurance**: you cannot know in advance which of ETS, ARIMA or Theta wins on your series, and the average is rarely the worst.
- Look at `Avg3` against `Avg4`. Adding one much weaker model drags the average toward it. An ensemble helps when its members are **about equally accurate and wrong in different ways**.
- The correlation matrix is the check. If two models' errors are near 1, the second one adds little. If one model is far worse than the rest, drop it rather than average it in.
- Everything above is scored on the same windows for every model, including the ensembles. An ensemble compared on different windows is not a comparison.

---
## Part 2. Weighted ensembles, scored honestly

A natural next move is to weight models by how good they were. Do it carelessly and you leak: weights picked on the windows you then score on will always look good.

The honest version splits the origins. **Calibration origins** set the weights; **evaluation origins** score everything.

- **Inverse-error weights**: $w_k \propto 1/\text{RMSE}_k$, normalized to sum to 1.
- **Median**: the middle forecast at each timestamp. Robust to one bad model, no weights to fit.

Fill in the `TODO`s with me.

In [ ]:
cutoffs = sorted(cv["cutoff"].unique())
calib = cv[cv["cutoff"].isin(cutoffs[:3])]      # first three origins: set the weights
evalu = cv[cv["cutoff"].isin(cutoffs[3:])].copy()  # last three origins: score

# TODO - RMSE of each of the four base models on the calibration origins.
rmse_cal = ...

# TODO - inverse-RMSE weights, normalized to sum to 1.
w = ...

print(w.round(3).to_string())

# TODO - weighted average and median of the four base models on the evaluation origins.
evalu["Weighted"] = ...
evalu["Median"] = ...

<details>
<summary><b>Show the filled-in lines</b></summary>

```python
rmse_cal = np.sqrt(((calib[names].sub(calib["y"], axis=0)) ** 2).mean())
w = (1 / rmse_cal) / (1 / rmse_cal).sum()

evalu["Weighted"] = (evalu[names] * w).sum(axis=1)
evalu["Median"] = evalu[names].median(axis=1)
```
</details>

In [ ]:
# Run after filling in the blanks above (or paste the solution).
evalu["Avg3"] = evalu[good].mean(axis=1)
evalu["Avg4"] = evalu[names].mean(axis=1)

cols = names + ["Avg3", "Avg4", "Weighted", "Median"]
e = evalu[cols].sub(evalu["y"], axis=0)
pd.DataFrame({"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean())}).round(1).sort_values("MAE")

**Questions to discuss.**

1. Did the weights recover the "drop Naive" decision that `Avg3` made by hand? What weight did `Naive` get?
2. Does `Weighted` beat `Avg4`? Beat `Avg3`? With only three calibration origins, how much would you trust the weights?
3. `Median` needs no fitted weights. When is that an advantage?
4. Stacking (a regression on the base forecasts) is the next step up. Why is it riskier than the average on a course-length project?

---
## Part 3. Intermittent demand

A series is intermittent when many periods have exactly zero demand. ETS and ARIMA bias toward zero or return a fractional "0.3 units a day" nobody can stock. MAPE is undefined. The new problem this part adds is that **MAE rewards forecasting zero.**

We simulate four daily SKUs so that the answer is known. Seed fixed, so everyone sees the same data.

In [ ]:
rng = np.random.default_rng(7)
n, H = 448, 56
dates = pd.date_range("2025-01-01", periods=n, freq="D")

def sparse(p, size):
    """Demand occurs with probability p; size drawn by `size(k)`."""
    occurs = rng.random(n) < p
    return np.where(occurs, size(n), 0).astype(float)

series = {
    "staple":       rng.poisson(8, n).astype(float),                                  # smooth
    "spare_part":   sparse(0.25, lambda k: 1 + rng.poisson(0.6, k)),                   # intermittent
    "capital_spare": sparse(0.12, lambda k: 1 + np.round(rng.lognormal(1.6, 0.9, k))), # lumpy
    "discontinued": sparse(0.30, lambda k: 1 + rng.poisson(0.6, k)),                   # stops selling
}
series["discontinued"][300:] = 0.0      # no demand after day 300

panel = pd.concat(
    [pd.DataFrame({"unique_id": k, "ds": dates, "y": v}) for k, v in series.items()],
    ignore_index=True,
)
tr = panel.groupby("unique_id", group_keys=False).apply(lambda d: d.iloc[:-H])
te = panel.groupby("unique_id", group_keys=False).apply(lambda d: d.iloc[-H:])

print("share of zero days:")
print((panel.assign(zero=panel["y"] == 0).groupby("unique_id")["zero"].mean()).round(2).to_string())

### Classify before you model

$$\text{ADI} = \frac{\text{periods}}{\text{periods with demand}}, \qquad \text{CV}^2 = \left(\frac{\sigma}{\mu}\right)^2 \text{ of the non-zero sizes}$$

Cut-offs 1.32 and 0.49 give smooth, erratic, intermittent, lumpy. Fill in the function.

In [ ]:
def classify(y):
    y = np.asarray(y, float)
    nz = y[y > 0]
    # TODO - ADI is total periods divided by periods with demand,
    # CV2 is (std / mean) ** 2 of the non-zero sizes. Use ddof=0 for the std.
    adi = ...
    cv2 = ...
    if adi < 1.32:
        kind = "smooth" if cv2 < 0.49 else "erratic"
    else:
        kind = "intermittent" if cv2 < 0.49 else "lumpy"
    return pd.Series({"ADI": round(adi, 2), "CV2": round(cv2, 2), "type": kind})

tr.groupby("unique_id")["y"].apply(lambda s: classify(s.values)).unstack()

<details>
<summary><b>Show the filled-in lines</b></summary>

```python
adi = len(y) / len(nz)
cv2 = (nz.std() / nz.mean()) ** 2
```

Your simulated series will not match its design label every time. `discontinued` looks like an ordinary intermittent series on the training window, and nothing in ADI or CV² tells you it has stopped selling.
</details>

In [ ]:
sf = StatsForecast(
    models=[CrostonClassic(), CrostonSBA(), TSB(alpha_d=0.2, alpha_p=0.2),
            ADIDA(), IMAPA(), HistoricAverage(), AutoETS(season_length=7)],
    freq="D",
)
fc = sf.forecast(df=tr, h=H)
fc["Zero"] = 0.0                       # the "forecast nothing" benchmark
ev = te.merge(fc, on=["unique_id", "ds"])
mods = [c for c in fc.columns if c not in ("unique_id", "ds")]

fc.groupby("unique_id").mean(numeric_only=True).round(2)

Look at the mean forecast per series. Most of these are fractions of a unit per day, which is the correct *rate* and not a stocking decision. Now score them.

In [ ]:
def scorecard(ev, mods):
    rows = []
    for k in mods:
        e = ev[k] - ev["y"]
        rows.append({
            "model": k,
            "MAE": e.abs().mean(),
            "RMSE": np.sqrt((e ** 2).mean()),
            "total_ratio": ev[k].sum() / max(ev["y"].sum(), 1e-9),   # forecast total / actual total
        })
    return pd.DataFrame(rows).round(2)

# Pool the three sparse series (the staple is not the point here).
sparse_ev = ev[ev["unique_id"] != "staple"]
scorecard(sparse_ev, mods).sort_values("MAE")

**Reading the table.**

- Sort by MAE and `Zero` sits at or near the top. MAE is minimized by the **median**, and the median of a mostly-zero series is zero. A forecast of nothing "wins" while being useless for stocking.
- Sort by RMSE instead and the ranking changes: squared error is minimized by the **mean**, which is the quantity you can plan with. That is why M5 used RMSSE.
- `total_ratio` is the check that cannot be gamed: forecast total over actual total across the horizon. `Zero` gets 0. A good rate forecast lands near 1.
- Do not report one metric. Report an RMSE-type score *and* the total ratio.

**Why TSB exists.** Look at the `discontinued` rows.

In [ ]:
disc = ev[ev["unique_id"] == "discontinued"]
print("actual demand in test window:", disc["y"].sum())
print("last 60 training days demand:", tr[tr.unique_id == "discontinued"]["y"].tail(60).sum())
disc[["CrostonClassic", "CrostonSBA", "TSB", "ADIDA", "IMAPA"]].mean().round(3)

The item has not sold for about 90 days, and Croston and SBA keep forecasting its last rate indefinitely, because they only update in periods with demand. TSB updates its probability of demand every period, so after a long run of zeros it decays toward zero. For spare parts and anything with a life cycle, that is the difference between carrying dead stock and not.

---
## Part 4. On your own

About 15 minutes. Work with the `ev`, `tr`, `te`, `fc` objects from Part 3.

**Tasks.**

1. **Ensemble.** Average `CrostonSBA`, `TSB` and `IMAPA`. Score it against its members on the pooled sparse series (RMSE and `total_ratio`). Check the error correlation between the three. Does the average earn its place?
2. **Dead stock.** For `discontinued`, plot `CrostonSBA` and `TSB` forecasts as a flat reference line against the actual (all zeros). Re-fit TSB with `alpha_p` of 0.05 and 0.4 and say what the parameter trades off.
3. **Stocking level.** The replenishment lead time is 7 days. For `spare_part`, the point rate times 7 is a *mean*, and stocking at the mean stocks out about half the time. Build an empirical 90% quantile of 7-day demand from the training history (sliding 7-day sums), compare it with `7 * rate`, and check on the test window how often each covers actual 7-day demand.
4. **Say it in one sentence** for a non-statistician: how many units of `spare_part` should the warehouse hold, and what is the chance of a stockout?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Solution sketch</b></summary>

```python
# 1. Ensemble
trio = ["CrostonSBA", "TSB", "IMAPA"]
sparse_ev = sparse_ev.copy()
sparse_ev["Trio"] = sparse_ev[trio].mean(axis=1)
print(scorecard(sparse_ev, trio + ["Trio"]))
print((sparse_ev[trio].sub(sparse_ev["y"], axis=0)).corr().round(2))

# 2. Dead stock
for ap in [0.05, 0.2, 0.4]:
    f = StatsForecast(models=[TSB(alpha_d=0.2, alpha_p=ap)], freq="D").forecast(
        df=tr[tr.unique_id == "discontinued"], h=H)
    print(ap, f["TSB"].mean().round(3))
# small alpha_p: slow to forget, stable when demand is genuinely sporadic
# large alpha_p: forgets fast, reacts to a stop but also to ordinary gaps

# 3. Stocking level
y_tr = tr[tr.unique_id == "spare_part"]["y"].to_numpy()
y_te = te[te.unique_id == "spare_part"]["y"].to_numpy()
LT = 7
sums_tr = np.convolve(y_tr, np.ones(LT), mode="valid")
q90 = np.quantile(sums_tr, 0.9)
mean_stock = LT * fc[fc.unique_id == "spare_part"]["CrostonSBA"].mean()
sums_te = np.convolve(y_te, np.ones(LT), mode="valid")
print("q90 stock:", q90, " covers:", (sums_te <= q90).mean().round(2))
print("mean stock:", round(mean_stock, 2), " covers:", (sums_te <= mean_stock).mean().round(2))
```

**What to notice.** The empirical quantile is a decision (units on the shelf with a stated stockout risk); the rate times lead time is a statistic. Overlapping 7-day windows are correlated, so the coverage check on 22 windows is rough. Say so.
</details>

---
## Wrap-up

1. **An average of different, similarly accurate models is insurance, not magic.** Check error correlation, drop the weak member, and score on the same windows.
2. **Weights must be learned on different origins than they are scored on.** Otherwise the weighted ensemble always looks best.
3. **Classify sparse series with ADI and CV² before choosing a model.** Use SBA or TSB, aggregate (ADIDA, IMAPA) when zeros dominate, and TSB when items die.
4. **MAE rewards forecasting zero.** Use an RMSE-type score plus total ratio, and prefer a quantile of lead-time demand to a rate.
5. **For the project**: if an ensemble or a fancier model only ties Seasonal Naive, run the Week 7 forecastability checks and write that up.